In [ ]:
import sys, json
from pathlib import Path
from types import SimpleNamespace
runners = list(Path('/kaggle/input').rglob('phase5_b1_train.py'))
guided = list(Path('/kaggle/input').rglob('phase5_sparse_guided_model.py'))
if len(runners) != 1 or len(guided) != 1: raise RuntimeError('Expected one Phase 1 B1 runner and one guided-window plan')
input_dir = str(runners[0].parent)
guided_dir = str(guided[0].parent)
sys.path[:0] = [input_dir + '/phase1_source', input_dir, guided_dir]
import phase5_b1_train as runner
import phase5_sparse_guided_model as guided_model
guided_model.install(guided_dir)
output_dir = Path('/kaggle/working/sparse_guided')
args = SimpleNamespace(input=input_dir, output=str(output_dir), epochs=4, batch=8, eval_batch=16, accum=2, lr=0.0002)
runner.train(args)
report = json.loads((output_dir / 'phase5_b1_lora_report.json').read_text())
report['method'] = 'B1 four-window bottleneck adapters with fit-only sparse-guided passage selection and label-wise smooth-max pooling'
report['window_count'] = 4
report['tokens_per_window'] = 192
report['selection_source'] = 'fit-only official-train 160k TF-IDF logistic coefficient salience; no dev/test labels'
(output_dir / 'phase5_b1_sparse_guided_report.json').write_text(json.dumps(report, indent=2))
(output_dir / 'phase5_b1_lora_prediction.zip').rename(output_dir / 'phase5_b1_sparse_guided_prediction.zip')
